In [1]:
import random
import os
import sys
from pathlib import Path
import contextlib
import render_generator_v2
import endpoint_generator_v2
import reduction_engine
import blame_design
from schemas import (
  validate_object_library,
  validate_assembly,
  v2_structural_gate
)
import shutil
import segment_analysis
import gwl_serializer
import itertools
import multiprocessing
import pathos.multiprocessing
from functools import partial
import json
import traceback

In [2]:
def generate_mushroom_entry(
    stem_d=3.0, 
    stem_h=6.0, 
    cap_d=6.0, 
    cap_h=2.0, 
    grid_size=None, 
    spacing=15.0
):
    if grid_size is None: grid_size = (10, 10)
    
    stem_center_z = stem_h / 2
    cap_center_z = stem_h + (cap_h / 2)
    job_id = f"mush_s{round(stem_d, 2)}_c{round(cap_d, 2)}_p{round(spacing, 2)}"

    return {
        "job_name": job_id,
        "prompt": (
            f"Create a {grid_size[0]}x{grid_size[1]} grid of mushroom-shaped micro-resonators. "
            f"Each unit consists of a vertical cylindrical stem {stem_d}um in diameter and {stem_h}um tall, "
            f"topped with a wide cylindrical cap {cap_d}um in diameter and {cap_h}um thick. "
            f"The structures are arrayed with a center-to-center spacing of {spacing}um."
        ),
        "objects": {
            "mushroom_stem": {
                "type": "geometry",
                "description": "Cylindrical support stem",
                "components": [
                    {
                        "type": "cylinder", 
                        "center": [0, 0, round(stem_center_z, 3)], 
                        "dimensions": {"diameter_um": stem_d, "z_um": stem_h}
                    }
                ]
            },
            "mushroom_cap": {
                "type": "geometry",
                "description": "Wide cylindrical cap",
                "components": [
                    {
                        "type": "cylinder", 
                        "center": [0, 0, round(cap_center_z, 3)], 
                        "dimensions": {"diameter_um": cap_d, "z_um": cap_h}
                    }
                ]
            },
            "mushroom": {
                "type": "geometry",
                "description": "Full mushroom assembly combining stem and cap",
                "components": [
                    {
                        "type": "cylinder", 
                        "center": [0, 0, round(stem_center_z, 3)], 
                        "dimensions": {"diameter_um": stem_d, "z_um": stem_h}
                    },
                    {
                        "type": "cylinder", 
                        "center": [0, 0, round(cap_center_z, 3)], 
                        "dimensions": {"diameter_um": cap_d, "z_um": cap_h}
                    }
                ]
            }
        },
        "assembly": {
            "type": "grid",
            "grid": {"x": grid_size[0], "y": grid_size[1]},
            "spacing_um": {"x": spacing, "y": spacing},
            "default_object": "mushroom"
        }
    }

In [3]:
def generate_gammadion_entry(
    center_w=2.0,
    center_l=6.0,
    arm_l=3.0,
    arm_w=1.5,
    thickness=1.0,
    spacing=15.0,
    grid_size=None,
    handedness="right"
):
    if grid_size is None: grid_size = (10, 10)
    
    h_mult = 1 if handedness == "right" else -1
    offset = center_l / 2
    arm_offset = (arm_l / 2) * h_mult
    z_center = thickness / 2

    # Component definitions for reuse
    cross_bars = [
        {"type": "box", "center": [0, 0, z_center], "dimensions": {"x_um": center_l, "y_um": center_w, "z_um": thickness}},
        {"type": "box", "center": [0, 0, z_center], "dimensions": {"x_um": center_w, "y_um": center_l, "z_um": thickness}}
    ]
    
    n_arm = {"type": "box", "center": [arm_offset, offset, z_center], "dimensions": {"x_um": arm_l, "y_um": arm_w, "z_um": thickness}}
    s_arm = {"type": "box", "center": [-arm_offset, -offset, z_center], "dimensions": {"x_um": arm_l, "y_um": arm_w, "z_um": thickness}}
    e_arm = {"type": "box", "center": [offset, -arm_offset, z_center], "dimensions": {"x_um": arm_w, "y_um": arm_l, "z_um": thickness}}
    w_arm = {"type": "box", "center": [-offset, arm_offset, z_center], "dimensions": {"x_um": arm_w, "y_um": arm_l, "z_um": thickness}}

    prompt_description = (
        f"Create a {handedness}-handed chiral gammadion array with a thickness of {thickness}um. "
        f"The central cross consists of two perpendicular bars, each {center_l}um long and {center_w}um wide. "
        f"Four chiral arms, each {arm_l}um long and {arm_w}um wide, are attached to the ends of the cross. "
        f"The units are arranged in a {grid_size[0]}x{grid_size[1]} grid with a spacing of {spacing}um."
    )

    return {
        "job_name": f"gammadion_{handedness}_{center_l}um",
        "prompt": prompt_description,
        "objects": {
            "central_cross": {
                "type": "geometry",
                "description": "The intersecting base bars",
                "components": cross_bars
            },
            "north_arm": {
                "type": "geometry",
                "description": "Chiral arm (+Y)",
                "components": [n_arm]
            },
            "south_arm": {
                "type": "geometry",
                "description": "Chiral arm (-Y)",
                "components": [s_arm]
            },
            "east_arm": {
                "type": "geometry",
                "description": "Chiral arm (+X)",
                "components": [e_arm]
            },
            "west_arm": {
                "type": "geometry",
                "description": "Chiral arm (-X)",
                "components": [w_arm]
            },
            "gammadion_unit": {
                "type": "geometry",
                "description": "Full gammadion assembly",
                "components": cross_bars + [n_arm, s_arm, e_arm, w_arm]
            }
        },
        "assembly": {
            "type": "grid",
            "grid": {"x": grid_size[0], "y": grid_size[1]},
            "spacing_um": {"x": spacing, "y": spacing},
            "default_object": "gammadion_unit"
        }
    }

In [4]:
def generate_srr_entry(
    outer_dim=5.0,
    width=1.0,
    gap_size=1.0,
    thickness=2.0,
    spacing=10.0,
    grid_size=None
):
    if grid_size is None: grid_size = (10, 10)
    
    # Standard center calculation for a box sitting on Z=0
    z_pos = thickness / 2
    edge_offset = outer_dim / 2 - width / 2
    inner_len = outer_dim - (2 * width)

    # Individual component definitions using the exact keys 
    # recognized by normalize_primitive_dimensions in reduction_engine.py
    top_bar = {
        "type": "box",
        "center": [0, edge_offset, z_pos],
        "dimensions": {"x_um": outer_dim, "y_um": width, "z_um": thickness}
    }
    bottom_bar = {
        "type": "box",
        "center": [0, -edge_offset, z_pos],
        "dimensions": {"x_um": outer_dim, "y_um": width, "z_um": thickness}
    }
    left_bar = {
        "type": "box",
        "center": [-edge_offset, 0, z_pos],
        "dimensions": {"x_um": width, "y_um": inner_len, "z_um": thickness}
    }

    prompt_description = (
        f"Create a square split-ring resonator (SRR) array in a {grid_size[0]}x{grid_size[1]} grid. "
        f"The resonator has an outer dimension of {outer_dim}um, a trace width of {width}um, and a thickness of {thickness}um. "
        f"It is composed of a top bar, a bottom bar, and a left connecting bar to form a 'C' shape with the gap on the right. "
        f"The units are spaced {spacing}um apart."
    )

    return {
        "job_name": f"srr_gap{gap_size}_w{width}",
        "prompt": prompt_description,
        "objects": {
            "srr_top_bar": {
                "type": "geometry",
                "description": "The top horizontal arm of the SRR",
                "components": [top_bar]
            },
            "srr_bottom_bar": {
                "type": "geometry",
                "description": "The bottom horizontal arm of the SRR",
                "components": [bottom_bar]
            },
            "srr_left_bar": {
                "type": "geometry",
                "description": "The vertical bar connecting the top and bottom arms",
                "components": [left_bar]
            },
            "srr_unit": {
                "type": "geometry",
                "description": "The complete split-ring resonator assembly",
                "components": [top_bar, bottom_bar, left_bar]
            }
        },
        "assembly": {
            "type": "grid",
            "grid": {"x": grid_size[0], "y": grid_size[1]},
            "spacing_um": {"x": spacing, "y": spacing},
            "default_object": "srr_unit"
        }
    }

In [5]:
def generate_dimer_entry(
    radius_um=1.0, 
    z_um=2.0, 
    gap_um=0.2, 
    shape_type="cylinder", 
    grid_size=(10, 10), 
    spacing_um=15.0
):
    """
    Generates a plasmonic dimer (two particles) split into named objects.
    """
    x_offset = radius_um + (gap_um / 2)
    z_center = z_um / 2
    
    # Define dimensions based on shapeF
    dims = {"diameter_um": radius_um * 2, "z_um": z_um} if shape_type == "cylinder" else \
           {"x_um": radius_um * 2, "y_um": radius_um * 2, "z_um": z_um}

    # Individual primitive definitions for geometry compliance
    left_comp = {
        "type": shape_type,
        "center": [-round(x_offset, 3), 0, z_center],
        "dimensions": dims
    }
    right_comp = {
        "type": shape_type,
        "center": [round(x_offset, 3), 0, z_center],
        "dimensions": dims
    }

    # Parametric prompt description
    prompt_description = (
        f"Create an array of {shape_type} dimers in a {grid_size[0]}x{grid_size[1]} grid. "
        f"Each particle has a {'diameter' if shape_type=='cylinder' else 'width'} of {radius_um*2}um and a height of {z_um}um. "
        f"The two particles are separated by a gap of {gap_um}um, centered at the origin. "
        f"The unit cells are spaced {spacing_um}um apart."
    )

    return {
        "job_name": f"dimer_{shape_type}_g{gap_um}",
        "prompt": prompt_description,
        "objects": {
            "left_particle": {
                "type": "geometry",
                "description": f"The left {shape_type} of the dimer pair",
                "components": [left_comp]
            },
            "right_particle": {
                "type": "geometry",
                "description": f"The right {shape_type} of the dimer pair",
                "components": [right_comp]
            },
            "dimer_unit": {
                "type": "geometry",
                "description": "The complete dimer assembly consisting of two adjacent particles",
                "components": [left_comp, right_comp]
            }
        },
        "assembly": {
            "type": "grid",
            "grid": {"x": grid_size[0], "y": grid_size[1]},
            "spacing_um": {"x": spacing_um, "y": spacing_um},
            "default_object": "dimer_unit"
        }
    }

In [6]:
def generate_cactus_entry(
    trunk_h=10.0, 
    trunk_d=1.0, 
    spine_l=3.0, 
    spine_d=0.4, 
    levels=3, 
    grid_size=None,
    spacing=20.0
):
    if grid_size is None: grid_size = (10, 10)
    
    spine_offset = (trunk_d / 2) + (spine_l / 2)
    level_spacing_z = trunk_h / (levels + 1)
    
    # Calculate all spine components for the final geometry
    spine_components = []
    for i in range(levels):
        z_pos = round((i + 1) * level_spacing_z, 3)
        
        # North/South spines (aligned with Y axis)
        spine_components.append({
            "type": "box", 
            "center": [0, round(spine_offset, 3), z_pos], 
            "dimensions": {"x_um": spine_d, "y_um": spine_l, "z_um": spine_d}
        })
        spine_components.append({
            "type": "box", 
            "center": [0, round(-spine_offset, 3), z_pos], 
            "dimensions": {"x_um": spine_d, "y_um": spine_l, "z_um": spine_d}
        })
        
        # East/West spines (aligned with X axis)
        spine_components.append({
            "type": "box", 
            "center": [round(spine_offset, 3), 0, z_pos], 
            "dimensions": {"x_um": spine_l, "y_um": spine_d, "z_um": spine_d}
        })
        spine_components.append({
            "type": "box", 
            "center": [round(-spine_offset, 3), 0, z_pos], 
            "dimensions": {"x_um": spine_l, "y_um": spine_d, "z_um": spine_d}
        })

    prompt_description = (
        f"Create a {grid_size[0]}x{grid_size[1]} array of 3D cactus resonators. "
        f"Each cactus has a central trunk {trunk_h}um tall and {trunk_d}um in diameter. "
        f"The design features {levels} levels of branching arms (spines), with 4 orthogonal arms per level. "
        f"Each arm is {spine_l}um long and {spine_d}um thick. "
        f"Units are arranged in a grid with {spacing}um spacing."
    )

    return {
        "job_name": f"cactus_h{trunk_h}_l{levels}",
        "prompt": prompt_description,
        "objects": {
            "cactus_trunk": {
                "type": "geometry",
                "description": "The main vertical stem of the cactus",
                "components": [
                    {
                        "type": "cylinder",
                        "center": [0, 0, round(trunk_h / 2, 3)],
                        "dimensions": {"diameter_um": trunk_d, "z_um": trunk_h}
                    }
                ]
            },
            "spine_arms": {
                "type": "geometry",
                "description": "The collection of horizontal radial arms",
                "components": spine_components
            },
            "full_cactus": {
                "type": "geometry",
                "description": "Complete cactus assembly with trunk and all spine levels",
                "components": [
                    {
                        "type": "cylinder",
                        "center": [0, 0, round(trunk_h / 2, 3)],
                        "dimensions": {"diameter_um": trunk_d, "z_um": trunk_h}
                    }
                ] + spine_components
            }
        },
        "assembly": {
            "type": "grid",
            "grid": {"x": grid_size[0], "y": grid_size[1]},
            "spacing_um": {"x": spacing, "y": spacing},
            "default_object": "full_cactus"
        }
    }

In [7]:
import json
import random

# --- [Insert the 5 Factory Functions here: Mushroom, Gammadion, SRR, Dimer, Cactus] ---

def generate_design_dataset(samples_per_type=200):
  dataset = []

  # 1. LOOP: MUSHROOMS (Vertical Stacking / Surface Waves)
  for _ in range(samples_per_type):
    s_d = round(random.uniform(1.0, 4.0), 2)
    # Constraint: Cap must be 1.5x to 3x wider than stem
    c_d = round(s_d * random.uniform(1.5, 3.0), 2)
    pitch = round(c_d + random.uniform(2.0, 8.0), 2)
    dataset.append(generate_mushroom_entry(stem_d=s_d, cap_d=c_d, spacing=pitch))

  # 2. LOOP: CHIRAL GAMMADIONS (Polarization / Chirality)
  for _ in range(samples_per_type):
    c_l = round(random.uniform(5.0, 10.0), 2)
    a_l = round(c_l * random.uniform(1.3, 2.3), 2)
    hand = random.choice(["right", "left"])
    pitch = round(c_l + a_l + 5.0, 2)
    thickness = round(random.uniform(2.0, 8.0), 2)
    dataset.append(generate_gammadion_entry(center_l=c_l, arm_l=a_l, handedness=hand, spacing=pitch, thickness = thickness))

  # # 3. LOOP: SPLIT-RING RESONATORS (Negative Permeability / Gaps)
  for _ in range(samples_per_type):
    o_dim = round(random.uniform(5.0, 12.0), 2)
    w = round(o_dim * 0.15, 2) # Maintain proportional trace width
    gap = round(random.uniform(0.5, 2.0), 2)
    pitch = round(o_dim + 8.0, 2)
    thickness = round(random.uniform(2.0, 8.0), 2)
    dataset.append(generate_srr_entry(outer_dim=o_dim, width=w, gap_size=gap, spacing=pitch, thickness = thickness))

  # # 4. LOOP: DIMERS (Plasmonic Hotspots)
  for _ in range(samples_per_type):
    r = round(random.uniform(3.5, 8.5), 2)
    g = round(random.uniform(0.5, 0.14), 2) # Sub-micron gaps are "real"
    h = round(3 + r * 3, 2)
    shape = random.choice(["cylinder", "box"])
    dataset.append(generate_dimer_entry(radius_um=r, z_um=h, gap_um=g, shape_type=shape))

  # 5. LOOP: 3D CACTI (Hierarchical 3D Branching)
  for _ in range(samples_per_type):
    h = round(random.uniform(10.0, 25.0), 2)
    lev = random.randint(2, 5)
    s_len = round(h * 0.25, 2)
    pitch = round((s_len * 2) + 10.0, 2)
    dataset.append(generate_cactus_entry(trunk_h=h, levels=lev, spine_l=s_len, spacing=pitch))


  # Validate dataset - fail immediately on any error.
  for design in dataset:
    v2_structural_gate(design)
    object_library_errors = validate_object_library({"objects": design.get("objects", {})})
    if len(object_library_errors) > 1:
      raise ValueError(f"Object library for design {design['job_name']} contained errors: {object_library_errors}\n\nDesign:\n{design}")
    object_names = list(design.get("objects", {}).keys())
    assembly_errors = validate_assembly({"assembly": design.get("assembly", {})}, object_names)
    if len(assembly_errors) > 1:
      raise ValueError(f"Assembly for design {design['job_name']} contained errors: {object_library_errors}\n\nDesign:\n{design}")
    
  

  return dataset

In [8]:
# Generate design dataset
metamaterials_dataset = generate_design_dataset(samples_per_type=1)
print(f"Success! Generated {len(metamaterials_dataset)} unique design/prompt pairs.")

[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
Success! Generated 5 unique design/prompt pairs.


In [9]:
def process_design(design: dict, print_params: dict, gwl_params: dict, dataset_dir = "Dataset"):
  output_dir = Path(dataset_dir, design['job_name'])
  if not output_dir.is_dir():
    output_dir.mkdir()

  # Dump design
  with open(output_dir / "design.json", "w") as f:
    json.dump(design, f, indent=2)

  # Dump print parameters
  with open(output_dir / "PrintParameters.txt", "w") as f:
    params = '\n'.join(f"{key}: {val}" for key, val in print_params.items())
    f.write(params + "\n")

  # Generate Renders
  render_generator_v2.generate_object_aware_renders(design, Path(output_dir, "Renders"))

  # Reduce primitives
  #recurse_dict(design)
  reduced = reduction_engine.reduce_assembly(design)
  reduction_errors = reduction_engine.validate_reduced_output(reduced)
  if len(reduction_errors) > 1:
    raise ValueError(f"Validation error reducting primitives for design {design['job_name']}: {object_library_errors}\n\nDesign:\n{design}")
  
  with open(output_dir / "reduced.json", "w") as f:
    json.dump(reduced, f, indent=2)

  # Generate endpoints
  endpoint_data = endpoint_generator_v2.generate_endpoint_json_v2(reduced, print_params)
  with open(output_dir / "output.json", 'w') as f:
    json.dump(endpoint_data, f, indent=2)

  # Generate GWL
  gwl_dir = output_dir / "GWL"
  gwl_files = gwl_serializer.generate_gwl_files(endpoint_data, gwl_params, gwl_dir)
  master_gwl = gwl_dir / f"{design['job_name']}_master.gwl"
  gwl_serializer.generate_master_gwl(gwl_files, gwl_params, master_gwl)

  # Run segment analysis
  segments_gen = segment_analysis.gwl_dir_segments_generator(gwl_dir)
  segments_analyzed = segment_analysis.analyze_segments(print_params, segments_gen)
  with open(output_dir / "segments_analyzed.json", "w") as f:
    json.dump(segments_analyzed, f, indent = 2)

  # Run blame design
  design_annotated = blame_design.blame_design(reduced, segments_analyzed, design, bbox_tol=0.1)
  with open(output_dir / "design_annotated.json", "w") as f:
    json.dump(design_annotated, f, indent = 2)

  return output_dir

In [ ]:
def mute_process():
  pass
  sys.stdout = open(os.devnull, 'w')

# Run pipeline for analysis
if __name__ == "__main__":
  if not Path("Dataset").is_dir():
    Path("Dataset").mkdir()
  else:
    shutil.rmtree(Path("Dataset")) ### Attention - behavior here is to delete old
    Path("Dataset").mkdir()
    
  
  # Distribute processing efficiently across cores
  with pathos.multiprocessing.Pool(processes = 8, initializer = mute_process) as pool: # processes = 8
    count = 0
    #print_params = endpoint_generator_v2.load_print_parameters("PrintParameters.txt")
    gwl_params = gwl_serializer.load_gwl_parameters("PrintParameters.txt")
    #p_func = partial(process_design, dataset_dir = "Dataset", print_params = print_params, gwl_params = gwl_params)
    
    def generate_tasks():
      rand = random.Random()
      for design in metamaterials_dataset:
        print_params = {'slice_distance_um': round(rand.uniform(0.05, 0.3), 2), #0.12,
                        'hatch_distance_um': round(rand.uniform(0.05, 0.3), 2), #0.14,
                        'voxel_xy_um': round(rand.uniform(0.05, 0.3), 2), #0.16,
                        'voxel_z_um': round(rand.uniform(0.05, 0.3), 2), #0.16,
                        'power_scaling': 0.6,
                        'laser_power': 40.0,
                        'scan_speed': 100.0,
                        'wait_time': 0.1,
                        'find_interface_at': 0.0,
                        'x_offset': 0.0,
                        'y_offset': 0.0,
                        'z_offset': 0.0}
        yield (design, print_params, gwl_params, "Dataset")
    
    for result in pool.starmap(process_design, generate_tasks(), chunksize=1):
      count = count + 1
      print(result, f"({count} / {len(metamaterials_dataset)} items)")

  [RENDER] central_cross: 3 views
  [RENDER] srr_top_bar: 3 views
  [RENDER] mushroom_stem: 3 views
  [RENDER] cactus_trunk: 3 views
  [RENDER] left_particle: 3 views
  [RENDER] north_arm: 3 views
  [RENDER] mushroom_cap: 3 views
  [RENDER] srr_bottom_bar: 3 views
  [RENDER] right_particle: 3 views
  [RENDER] spine_arms: 3 views
  [RENDER] south_arm: 3 views
  [RENDER] mushroom: 3 views
  [RENDER] dimer_unit: 3 views
  [RENDER] full_cactus: 3 views
  [RENDER] srr_left_bar: 3 views
  [RENDER] east_arm: 3 views
  [RENDER] final_assembly: 3 views
  [RENDER] final_assembly: 3 views[Endpoint Generator V2 - Shapely Mode]

[Endpoint Generator V2 - Shapely Mode]  Processing 200 primitives...

  Processing 200 primitives...
  [RENDER] srr_unit: 3 views
  [RENDER] west_arm: 3 views
  [RENDER] final_assembly: 3 views
[Endpoint Generator V2 - Shapely Mode]
  Processing 900 primitives...
  [RENDER] gammadion_unit: 3 views
  [RENDER] final_assembly: 3 views
[Endpoint Generator V2 - Shapely Mode]
  P

In [ ]:
### Validate generated dataset
d_dir = Path("Dataset")
for entry in os.listdir(d_dir):
  if not os.path.isfile(d_dir / entry / "design.json"):
    print(f"{entry} has no design.json")
  if not os.path.isfile(d_dir / entry / "output.json"):
    print(f"{entry} has no output.json (endpoint_generator_v2.py results)")
  if not os.path.isfile(d_dir / entry / "reduced.json"):
    print(f"{entry} has no reduced.json (reduction_engine.py results)")
  if not os.path.isfile(d_dir / entry / "segments_analyzed.json"):
    print(f"{entry} has no segments_analyzed.json (segment_analysis.py results)")
  if not os.path.isfile(d_dir / entry / "design_annotated.json"):
    print(f"{entry} has no design_annotated.json (blame_design.py results)")
  if not os.path.isfile(d_dir / entry / "PrintParameters.txt"):
    print(f"{entry} has no PrintParameters.txt (customized parameters for entry)")
  if not os.path.isdir(d_dir / entry / "Renders") or len(os.listdir(d_dir / entry / "Renders")) == 0:
    print(f"{entry} has no Renders directory or it is empty (render_generator_v2.py results)")
  if not os.path.isdir(d_dir / entry / "GWL") or len(os.listdir(d_dir / entry / "GWL")) == 0:
    print(f"{entry} has no GWL directory or it is empty (gwl_serializer.py results)")  


# Qdrant Dataset Generation

In [18]:
from qdrant_client import QdrantClient # 1.17.0
from qdrant_client.models import PointStruct, VectorParams, Datatype, Distance
from langchain_ollama import OllamaEmbeddings
import json

In [19]:
embeddings_model = OllamaEmbeddings(model="mxbai-embed-large", base_url="http://127.0.0.1:11434")

In [20]:
points = []
for listing in os.listdir(Path("Dataset")):
  project_dir = Path("Dataset", listing)
  with open(project_dir / "design.json", 'r') as f:
    design = json.load(f)
  with open(project_dir / "segments_analyzed.json", 'r') as f:
    segments_analyzed = json.load(f)
  
  embedding = embeddings_model.embed_query(design["prompt"])  # Compute embedding

  # We lack good error analysis right now - guess vaguely for the model.
  if segments_analyzed["failed_segments"] == 0:
    problems = []
  else:
    problems = ["Contains traces that are not adhered (likely due to a floating object or overhang)"]
        
  points.append(
    PointStruct(
      id=hash(design["prompt"]),  # Unique ID for the point
      vector=embedding,  # The embedding vector
      payload={"prompt": design["prompt"], "design": design, "printable": segments_analyzed["failed_segments"] == 0, "problems": problems}
    )
  )

In [21]:
# Create qdrant vector store
qclient = QdrantClient(path = './Dataset/qdrant_dataset')
qclient.create_collection(
    collection_name="prompts",
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE,
        datatype=Datatype.FLOAT32
    )
)

True

In [22]:
# Upload points
qclient.upsert(collection_name="prompts", points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

# Example querying

In [41]:
with open("prompt.txt", 'r') as f:
  embedding = embeddings_model.embed_query(f.read())
search_results = qclient.query_points(
    collection_name="prompts",
    query=embedding,
    with_payload=True,
    limit=8
).points

In [47]:
successful_example = None
failed_example = None
# Try to pick one successful example and one failed example
for result in search_results:
  if result.payload["printable"] == True:
    successful_example = result.payload
    break

for result in search_results:
  if result.payload["printable"] == False:
    failed_example = result.payload
    break

In [48]:
successful_example

{'prompt': 'Construct an array of cylinder dimers. Each particle has a diameter of 2.18um. The critical gap between the two particles is 0.71um. Arrange in a 10x10 grid.',
 'design': {'job_name': 'dimer_cylinder_g0.71',
  'prompt': 'Construct an array of cylinder dimers. Each particle has a diameter of 2.18um. The critical gap between the two particles is 0.71um. Arrange in a 10x10 grid.',
  'objects': {'dimer_unit': {'type': 'geometry',
    'components': [{'type': 'cylinder',
      'center': [-1.445, 0, 1.635],
      'dimensions': {'diameter_um': 2.18, 'height_um': 3.27}},
     {'type': 'cylinder',
      'center': [1.445, 0, 1.635],
      'dimensions': {'diameter_um': 2.18, 'height_um': 3.27}}]}},
  'assembly': {'type': 'grid',
   'grid': {'x': 10, 'y': 10},
   'spacing_um': {'x': 15.0, 'y': 15.0},
   'default_object': 'dimer_unit'}},
 'printable': True,
 'problems': []}

In [49]:
failed_example

{'prompt': 'Grad- Create an array of nailhead-style structures. Each structure consists of two stacked cylinders: a base cylinder that is 6 um tall and 3 um in diameter, and a top cylinder centered on the base that is 2 um tall and 6 um in diameter. Arranged in a 10x10 square array with 15 um spacing.',
 'design': {'metadata': {'category': '',
   'timestamp': '20260215_013906',
   'model': 'qwen2.5-coder:3b',
   'tokens': {'prompt_tokens': 1287,
    'completion_tokens': 326,
    'total_tokens': 1613},
   'errors': []},
  'prompt': 'Grad- Create an array of nailhead-style structures. Each structure consists of two stacked cylinders: a base cylinder that is 6 um tall and 3 um in diameter, and a top cylinder centered on the base that is 2 um tall and 6 um in diameter. Arranged in a 10x10 square array with 15 um spacing.',
  'job_name': 'nailheads_array',
  'objects': {'base_nailhead_cylinder': {'type': 'geometry',
    'description': 'Base nailhead cylinder',
    'components': [{'type': 'c

# Evaluating The Model Against The Dataset